## Machine Translation And Question Answering using Pretrained Models

Modern Natural Language Processing (NLP) models can perform multiple language tasks without training from scratch. In this notebook tutorial, we will learn how to use pretrained transformer models for Machine Translation (English ↔ other languages) and Question Answering (answering questions from text). We will use `MarianMT`, a model specialized for translation and `T5` (Text-To-Text Transfer Transformer), a flexible model that can perform many NLP tasks using a single format.

### What Are We Going to Cover?

In this notebook, we will:

- Understand what `MarianMT` and `T5` models are

- Install and set up Hugging Face Transformers

- Use `MarianMT` for machine translation

- Use `T5` for question answering

- Understand tokenization and text generation

- Create reusable helper functions

- Compare task-specific vs multi-task models

### Key Learnings

By the end of this tutorial, learners will be able to:

- Use pretrained translation models without training

- Translate text between languages

- Perform question answering using context passages

- Understand how text-generation models work

- Apply the same workflow to other NLP tasks


### Setup and Installation
#### Install Required Libraries

In [14]:
!pip install -U transformers sentencepiece torch


#### Import Libraries

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

### Understanding the Models

#### **MarianMT**

MarianMT is a neural machine translation model that is optimised specifically for translation. Each model version handles one language pair (e.g., English → French).

#### **T5**

T5 is a text-to-text model. Using this model every task is framed as text input → text output. The same model can do translation, question answering,summarisation, text classification.

### Machine Translation using MarianMT
In this section, we will use `Helsinki-NLP/opus-mt-en-fr`, pretrained model to translate from English to French using MarianMT.

In [16]:
model_name = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model.eval()

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


MarianMTModel(
  (model): MarianModel(
    (shared): Embedding(59514, 512, padding_idx=59513)
    (encoder): MarianEncoder(
      (embed_tokens): Embedding(59514, 512, padding_idx=59513)
      (embed_positions): MarianSinusoidalPositionalEmbedding(512, 512)
      (layers): ModuleList(
        (0-5): 6 x MarianEncoderLayer(
          (self_attn): MarianAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation_fn): SiLU()
          (fc1): Linear(in_features=512, out_features=2048, bias=True)
          (fc2): Linear(in_features=2048, out_features=512, bias=True)
          (final_layer_norm): LayerNorm((512,), eps=1e-05

#### Translate a Sentence

###### **Understanding Tokenization: From Text to Token IDs**

Before we perform inference, let us understand how the tokenizer converts raw text into a format the model can process.

**How Text is Converted to Token IDs:**

1. **Tokenization**: The input text is split into smaller units called *tokens*. These can be words, subwords, or characters depending on the tokenizer. For example, "transforming" might be split into ["transform", "ing"].

2. **Vocabulary Mapping**: Each token is mapped to a unique integer ID from the model's vocabulary. For instance, "machine" → 1234, "learning" → 5678.

3. **Tensor Conversion**: The sequence of token IDs is converted into a PyTorch tensor that can be fed into the model.

**Special Tokens Added by the Tokenizer:**

Tokenizers automatically add special tokens to help the model understand the structure of the input:

| Token | Purpose |
|-------|---------|
| `<s>` or `[CLS]` | Marks the beginning of a sequence |
| `</s>` or `[SEP]` | Marks the end of a sequence or separates segments |
| `<pad>` | Padding token to make all sequences the same length |
| `<unk>` | Represents unknown/out-of-vocabulary words |

When we call `tokenizer(text, return_tensors="pt")`, the tokenizer returns a dictionary containing:
- `input_ids`: The token ID sequence (including special tokens)
- `attention_mask`: Indicates which tokens are real (1) vs padding (0)

Let us see this in action:

In [17]:
# Let's examine the tokenization process step by step
sample_text = "Machine learning is transforming the world."

# Tokenize the text
encoded = tokenizer(sample_text, return_tensors="pt", padding=True)

print("Original Text:", sample_text)
print("\n--- Tokenization Output ---")
print("Input IDs:", encoded["input_ids"])
print("Attention Mask:", encoded["attention_mask"])

# Convert token IDs back to tokens to see individual tokens
tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
print("\nTokens:", tokens)

# Show the special tokens used by this tokenizer
print("\n--- Special Tokens ---")
print(f"EOS Token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"PAD Token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

Original Text: Machine learning is transforming the world.

--- Tokenization Output ---
Input IDs: tensor([[12794,  3655,    32, 33846,     4,   522,     3,     0]])
Attention Mask: tensor([[1, 1, 1, 1, 1, 1, 1, 1]])

Tokens: ['▁Machine', '▁learning', '▁is', '▁transforming', '▁the', '▁world', '.', '</s>']

--- Special Tokens ---
EOS Token: </s> (ID: 0)
PAD Token: <pad> (ID: 59513)


Now that we understand how tokenization works, let us perform the actual translation. The tokenized input (token IDs) is passed to the model, which generates output token IDs that are then decoded back into readable text:

In [18]:
text = "Machine learning is transforming the world."

inputs = tokenizer(text, return_tensors="pt", padding=True)

with torch.no_grad():
    translated_tokens = model.generate(**inputs)

translation = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
translation

"L'apprentissage automatique transforme le monde."

#### Creating a Reusable Translation Function
Let us create a modular translation function

In [19]:
def translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model.generate(**inputs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [20]:
translate("Artificial intelligence is the future of technology.")

"L'intelligence artificielle est l'avenir de la technologie."

When using MarianMT for machine translation, the source text is tokenized into numbers. The encoder understands the source language and the decoder generates translated text token by token. Finally, output tokens are converted back into readable text.

### Question Answering using T5
Next, we will use T5 model for question answer generation task. T5 treats QA as a text generation problem:
```
Input:
question: What is AI?
context: Artificial intelligence is a field of computer science...

Output:
Artificial intelligence is a field of computer science.

```

Let us now load a pretrained model.

In [21]:
t5_model_name = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(t5_model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_model_name)

t5_model.eval()

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

Next, we need to prepare question and context set.

In [22]:
question = "What is deep learning?"

context = """
Deep learning is a subset of machine learning that uses neural networks
with many layers. It is particularly effective for image and speech recognition.
"""


Let us next format input for T5

In [23]:
input_text = f"question: {question} context: {context}"

inputs = t5_tokenizer(
    input_text,
    return_tensors="pt",
    truncation=True,
    padding=True
)


Let us generate the answers next.

In [24]:
with torch.no_grad():
    output = t5_model.generate(**inputs, max_length=50)

answer = t5_tokenizer.decode(output[0], skip_special_tokens=True)
answer


'a subset of machine learning that uses neural networks with many layers'

#### Modularised QA Generation Function

In [25]:
def answer_question(question, context):
    input_text = f"question: {question} context: {context}"
    inputs = t5_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = t5_model.generate(**inputs, max_length=64)

    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)


In [26]:
answer_question(
    "What is deep learning?",
    context
)


'a subset of machine learning that uses neural networks with many layers'

Text generation in `T5` uses an encoder-decoder model. The encoder understands the input text and the decoder generates the output text. The words are converted to tokens (pieces of the words) and in the generation process, it is a auto-regressive decoding that happens, which means that the current word is used for predicting the next word.

While MarianMT is used for only translation tasks, T5 is used for multi-task learning. MarianMT can be used for production level translation and is faster than T5. However, T5 is more general purpose NLP model.

### Conclusion

In this notebook, we used MarianMT for machine translation and T5 for question answering. We learned how text-generation models work and built reusable inference pipelines. We applied pretrained transformers to real NLP tasks. This notebook also demonstrates how one model architecture can solve multiple NLP problems.